### SFT Baseline — SCAN Length Split

Train HF causal LMs on the SCAN length generalization task with regularization ablations:
1. **SFT** — pure supervised fine-tuning
2. **SFT + Label Smoothing** — softens target distribution
3. **SFT + Dropout** — increases hidden/attention dropout
4. **SFT + MBE** — matrix-based entropy regularization on hidden states
5. **SFT + Frobenius** — Frobenius norm regularization on hidden states
6. **SFT + Spectral Norm** — spectral normalization on linear layers (Lipschitz constraint)

**Usage:** Run cells 1–4 (setup), pick a config in cell 5, train, then eval.

In [ ]:
# Setup
import os
import torch
import numpy as np
import random
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from src.mbe import patch_mbe, patch_frobenius
from src.gapt_trainer import aggregate_log_history
from train_cpt_scan import (
    load_scan_length_manual,
    compute_scan_accuracy,
    check_scan_match,
    ScanAccuracyCallback,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class RegTrainer(Trainer):
    """Trainer with optional hidden-state regularization (MBE or Frobenius).
    Always computes MBE on eval to track representation entropy over training."""

    def __init__(self, reg_type="none", reg_weight=1.0, patch_size=4, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.reg_type = reg_type      # "none", "mbe", "frobenius"
        self.reg_weight = reg_weight
        self.patch_size = patch_size
        self._last_ce_loss = None
        self._last_reg_loss = None
        self._last_mbe_val = None     # MBE value (always computed, even if not used as loss)
        self._eval_ce_losses = []
        self._eval_mbe_vals = []

    def _compute_mbe(self, hidden_states, ce_device):
        """Compute mean MBE across layers (skip first/last)."""
        num_layers = len(hidden_states)
        layer_mask = torch.zeros(num_layers, device=ce_device)
        if num_layers > 2:
            layer_mask[1:-1] = 1.0
        else:
            layer_mask[:] = 1.0
        vals = []
        for h in hidden_states:
            B, S, D = h.shape
            if S % self.patch_size != 0:
                h = h[:, :S - (S % self.patch_size), :]
            vals.append(patch_mbe(h, self.patch_size).float())
        per_layer = torch.stack(vals)
        masked = per_layer * layer_mask
        denom = layer_mask.sum()
        return masked.sum() / denom if denom > 0 else torch.tensor(0.0, device=ce_device)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        if "labels" not in inputs:
            inputs["labels"] = inputs["input_ids"]

        # Always request hidden states during eval (for MBE tracking)
        need_hidden = (self.reg_type != "none" and model.training) or (not model.training)
        outputs = model(**inputs, output_hidden_states=need_hidden, return_dict=True)

        if self.label_smoother is not None and "labels" in inputs:
            ce_loss = self.label_smoother(outputs, inputs["labels"])
        else:
            ce_loss = outputs.loss

        reg_loss = torch.tensor(0.0, device=ce_loss.device)
        mbe_val = torch.tensor(0.0, device=ce_loss.device)

        if need_hidden and outputs.hidden_states is not None:
            hidden_states = outputs.hidden_states[1:]  # skip embedding layer
            # Always compute MBE for tracking
            mbe_val = self._compute_mbe(hidden_states, ce_loss.device)

            if self.reg_type != "none" and model.training:
                if self.reg_type == "mbe":
                    reg_loss = mbe_val
                elif self.reg_type == "frobenius":
                    # Frobenius uses its own function
                    num_layers = len(hidden_states)
                    layer_mask = torch.zeros(num_layers, device=ce_loss.device)
                    if num_layers > 2:
                        layer_mask[1:-1] = 1.0
                    else:
                        layer_mask[:] = 1.0
                    frob_vals = []
                    for h in hidden_states:
                        B, S, D = h.shape
                        if S % self.patch_size != 0:
                            h = h[:, :S - (S % self.patch_size), :]
                        frob_vals.append(patch_frobenius(h, self.patch_size).float())
                    frob_per_layer = torch.stack(frob_vals)
                    masked = frob_per_layer * layer_mask
                    denom = layer_mask.sum()
                    if denom > 0:
                        reg_loss = masked.sum() / denom

        final_loss = ce_loss + self.reg_weight * reg_loss

        self._last_ce_loss = ce_loss.detach()
        self._last_reg_loss = reg_loss.detach()
        self._last_mbe_val = mbe_val.detach()
        return (final_loss, outputs) if return_outputs else final_loss

    def log(self, logs, start_time=None):
        logs = dict(logs)
        is_eval_log = any(k.startswith("eval_") for k in logs)
        if not is_eval_log:
            if self._last_ce_loss is not None:
                logs.setdefault("ce_loss", self._last_ce_loss.item())
            if self._last_reg_loss is not None:
                logs.setdefault("reg_loss", self._last_reg_loss.item())
            if self._last_mbe_val is not None:
                logs.setdefault("mbe_val", self._last_mbe_val.item())
        if self.state.epoch is not None:
            logs["epoch"] = round(self.state.epoch, 2)
        if self.state.global_step is not None:
            logs["step"] = self.state.global_step
        self.state.log_history.append(logs)

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        loss, logits, labels = super().prediction_step(
            model, inputs, prediction_loss_only, ignore_keys=ignore_keys
        )
        if self._last_ce_loss is not None:
            self._eval_ce_losses.append(self._last_ce_loss.float().cpu())
        if self._last_mbe_val is not None:
            self._eval_mbe_vals.append(self._last_mbe_val.float().cpu())
        return loss, logits, labels

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        self._eval_ce_losses = []
        self._eval_mbe_vals = []
        metrics = super().evaluate(eval_dataset=eval_dataset, ignore_keys=ignore_keys, metric_key_prefix=metric_key_prefix)
        extra = {}
        if self._eval_ce_losses:
            extra[f"{metric_key_prefix}_ce_loss"] = torch.stack(self._eval_ce_losses).mean().item()
        if self._eval_mbe_vals:
            extra[f"{metric_key_prefix}_mbe_val"] = torch.stack(self._eval_mbe_vals).mean().item()
        if extra:
            self.log(extra)
            metrics.update(extra)
        return metrics

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Using device: cpu


In [ ]:
# Load SCAN Dataset (length split)
SEED = 42
MAX_LENGTH = 128

dataset = load_scan_length_manual()

def format_scan(example):
    return {"text": f"Input: {example['commands']}\nOutput: {example['actions']}"}

dataset = dataset.map(format_scan)

full_train = dataset["train"]
train_test_split = full_train.train_test_split(test_size=0.1, seed=SEED)
train_data = train_test_split["train"]
test_id = train_test_split["test"]
test_ood = dataset["test"]
if len(test_ood) > 500:
    test_ood = test_ood.shuffle(seed=SEED).select(range(500))

print(f"Train: {len(train_data)} | Test ID: {len(test_id)} | Test OOD: {len(test_ood)}")
print(f"Example: {train_data[0]['text'][:120]}")

Map:   0%|          | 0/16990 [00:00<?, ? examples/s]

Map:   0%|          | 0/3920 [00:00<?, ? examples/s]

Train: 15291 | Test ID: 1699 | Test OOD: 500
Example: Input: walk opposite left and turn opposite right thrice
Output: I_TURN_LEFT I_TURN_LEFT I_WALK I_TURN_RIGHT I_TURN_RIGH


In [ ]:
# Model Init + Tokenization
MODEL_NAME = "Qwen/Qwen3-0.6B"

def init_model(model_name=MODEL_NAME, dropout=0.0):
    """Load a fresh HF causal LM + tokenizer, optionally with increased dropout."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.bfloat16, device_map="auto",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = model.config.eos_token_id
    # Override dropout if requested
    if dropout > 0:
        for name, module in model.named_modules():
            if isinstance(module, torch.nn.Dropout):
                module.p = dropout
    return model, tokenizer

def apply_spectral_norm(model):
    """Apply spectral normalization to all Linear layers (except lm_head)."""
    from torch.nn.utils import spectral_norm
    count = 0
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear) and "lm_head" not in name:
            try:
                spectral_norm(module, name="weight")
                count += 1
            except Exception:
                pass
    print(f"Applied spectral norm to {count} Linear layers")
    return model

def tokenize_scan(data, tokenizer, max_length=MAX_LENGTH):
    """Tokenize SCAN with answer-only labels (mask question with -100)."""
    def _tokenize(examples):
        texts = examples["text"]
        tokenized = tokenizer(
            texts, padding="max_length", truncation=True, max_length=max_length,
        )
        labels = []
        for i, text in enumerate(texts):
            input_ids = tokenized["input_ids"][i]
            split_marker = "\nOutput: "
            answer_start = text.find(split_marker)
            if answer_start != -1:
                header_text = text[:answer_start + len(split_marker)]
                mask_len = len(tokenizer(header_text, add_special_tokens=False)["input_ids"])
                label = [-100] * mask_len + input_ids[mask_len:]
                label = label[:max_length] + [-100] * max(0, max_length - len(label))
            else:
                label = input_ids.copy()
            labels.append(label)
        tokenized["labels"] = labels
        return tokenized

    tok_data = data.map(_tokenize, batched=True, remove_columns=data.column_names)
    tok_data.set_format("torch")
    return tok_data

print(f"Model: {MODEL_NAME}")

---
## Config Selection

Pick a configuration: `"sft"`, `"sft_ls"`, `"sft_dropout"`, `"sft_mbe"`, `"sft_frob"`, or `"sft_spectral"`.

In [ ]:
# Training Configuration
CONFIG = "sft"  # Choose: "sft", "sft_ls", "sft_dropout", "sft_mbe", "sft_frob", "sft_spectral"

EPOCHS = 3
LR = 5e-5
BATCH_SIZE = 8
PATCH_SIZE = 4
REG_WEIGHT = 1.0
ACC_EVAL_STEPS = 100000  # Only evaluate at the end of training
ACC_EVAL_SAMPLES = 200

config_map = {
    "sft":          dict(label_smoothing=0.0, dropout=0.0, reg_type="none",      reg_weight=0.0, spectral=False),
    "sft_ls":       dict(label_smoothing=0.1, dropout=0.0, reg_type="none",      reg_weight=0.0, spectral=False),
    "sft_dropout":  dict(label_smoothing=0.0, dropout=0.1, reg_type="none",      reg_weight=0.0, spectral=False),
    "sft_mbe":      dict(label_smoothing=0.0, dropout=0.0, reg_type="mbe",       reg_weight=REG_WEIGHT, spectral=False),
    "sft_frob":     dict(label_smoothing=0.0, dropout=0.0, reg_type="frobenius", reg_weight=REG_WEIGHT, spectral=False),
    "sft_spectral": dict(label_smoothing=0.0, dropout=0.0, reg_type="none",      reg_weight=0.0, spectral=True),
}

cfg = config_map[CONFIG]
print(f"Config: {CONFIG} → {cfg}")

In [ ]:
# Initialize Model + Tokenize + Build Trainer
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

model, tokenizer = init_model(dropout=cfg["dropout"])

if cfg.get("spectral", False):
    model = apply_spectral_norm(model)

tok_train = tokenize_scan(train_data, tokenizer)
tok_id = tokenize_scan(test_id, tokenizer)
tok_ood = tokenize_scan(test_ood, tokenizer)

acc_callback = ScanAccuracyCallback(
    tokenizer=tokenizer,
    test_id=test_id,
    test_ood=test_ood,
    eval_steps=ACC_EVAL_STEPS,
    max_samples=ACC_EVAL_SAMPLES,
)

output_dir = f"./ckpt/scan_sft_{CONFIG}"

trainer = RegTrainer(
    reg_type=cfg["reg_type"],
    reg_weight=cfg["reg_weight"],
    patch_size=PATCH_SIZE,
    model=model,
    train_dataset=tok_train,
    eval_dataset={"id": tok_id, "ood": tok_ood},
    args=TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        logging_steps=20,
        logging_first_step=True,
        eval_strategy="steps",
        eval_steps=100,
        label_smoothing_factor=cfg["label_smoothing"],
        save_strategy="no",
        eval_on_start=True,
        report_to="none",
        bf16=True,
        dataloader_pin_memory=False,
        seed=SEED,
    ),
    callbacks=[acc_callback],
)

print(f"Trainer ready: {CONFIG} | {len(tok_train)} train samples | model={MODEL_NAME}")

In [ ]:
trainer.train()

In [ ]:
# Final Accuracy Evaluation
model.eval()
acc_id = compute_scan_accuracy(model, tokenizer, test_id, max_samples=200)
acc_ood = compute_scan_accuracy(model, tokenizer, test_ood, max_samples=200)
print(f"Config: {CONFIG}")
print(f"  ID Accuracy:  {acc_id:.2%}")
print(f"  OOD Accuracy: {acc_ood:.2%}")

In [ ]:
# Plot Training Curves
df = aggregate_log_history(trainer.state.log_history)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle(f"SFT on SCAN — {CONFIG} ({MODEL_NAME})", fontsize=14)

for ax, (key, title) in zip(axes, [("loss", "Train Loss"), ("mbe_val", "Train MBE"), ("eval_id_acc", "ID Accuracy"), ("eval_ood_acc", "OOD Accuracy")]):
    if key in df.columns and df[key].notna().any():
        ax.plot(df["step"], df[key], linewidth=0.8)
        if "acc" in key:
            ax.set_ylim(-0.05, 1.05)
    ax.set_title(title); ax.set_xlabel("step"); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# Inspect OOD Predictions
model.eval()
n_samples = 5

print("=" * 80)
print(f"OOD Predictions ({CONFIG})")
print("=" * 80)

for i in range(min(n_samples, len(test_ood))):
    example = test_ood[i]
    full_text = example["text"]
    input_part = full_text.split("\nOutput:")[0] + "\nOutput:"
    target_action = full_text.split("\nOutput: ")[1].strip()

    inputs = tokenizer(input_part, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=128, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    prediction = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    is_correct = check_scan_match(target_action, prediction)

    print(f"\n--- Sample {i+1} {'✓' if is_correct else '✗'} ---")
    print(f"Input:     {input_part}")
    print(f"Target:    {target_action}")
    print(f"Predicted: {prediction[:200]}")

model.train();

In [ ]:
# ============================================================
# Eval MBE Trend (does MBE increase or decrease on val data?)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Eval MBE Over Training — {CONFIG} ({MODEL_NAME})", fontsize=14)

for ax, prefix, title in [
    (axes[0], "eval_id", "ID (in-distribution)"),
    (axes[1], "eval_ood", "OOD (length generalization)"),
]:
    mbe_col = f"{prefix}_mbe_val"
    ce_col = f"{prefix}_ce_loss"
    if mbe_col in df.columns and df[mbe_col].notna().any():
        mbe_vals = df[mbe_col].dropna()
        steps = df.loc[mbe_vals.index, "step"]
        ax.plot(steps, mbe_vals, "o-", color="red", linewidth=1.5, markersize=4, label="MBE")
        
        ax.annotate(f"{mbe_vals.iloc[0]:.4f}", xy=(steps.iloc[0], mbe_vals.iloc[0]),
                    fontsize=8, color="red", ha="left", va="bottom")
        ax.annotate(f"{mbe_vals.iloc[-1]:.4f}", xy=(steps.iloc[-1], mbe_vals.iloc[-1]),
                    fontsize=8, color="red", ha="right", va="bottom")
        
        delta = mbe_vals.iloc[-1] - mbe_vals.iloc[0]
        direction = "↑ increased" if delta > 0 else "↓ decreased"
        ax.set_title(f"{title}\nMBE {direction} by {abs(delta):.4f}")
    else:
        ax.set_title(f"{title} (no MBE data)")
    
    if ce_col in df.columns and df[ce_col].notna().any():
        ax2 = ax.twinx()
        ce_vals = df[ce_col].dropna()
        ce_steps = df.loc[ce_vals.index, "step"]
        ax2.plot(ce_steps, ce_vals, "--", color="blue", linewidth=0.8, alpha=0.5, label="CE")
        ax2.set_ylabel("CE Loss", color="blue", fontsize=9)
        ax2.tick_params(axis="y", labelcolor="blue")
    
    ax.set_xlabel("step")
    ax.set_ylabel("MBE", color="red")
    ax.tick_params(axis="y", labelcolor="red")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
---
## Compare All Configs

Run the cell below to train all configs sequentially and compare.

In [ ]:
# Run All Configs (sequential comparison)
results = {}
all_dfs = {}

for config_name, cfg in config_map.items():
    print(f"\n{'='*60}")
    print(f"Training: {config_name} → {cfg}")
    print(f"{'='*60}")

    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model, tokenizer = init_model(dropout=cfg["dropout"])
    
    if cfg.get("spectral", False):
        model = apply_spectral_norm(model)
    
    tok_train = tokenize_scan(train_data, tokenizer)
    tok_id = tokenize_scan(test_id, tokenizer)
    tok_ood = tokenize_scan(test_ood, tokenizer)

    acc_cb = ScanAccuracyCallback(
        tokenizer=tokenizer, test_id=test_id, test_ood=test_ood,
        eval_steps=ACC_EVAL_STEPS, max_samples=ACC_EVAL_SAMPLES,
    )

    t = RegTrainer(
        reg_type=cfg["reg_type"],
        reg_weight=cfg["reg_weight"],
        patch_size=PATCH_SIZE,
        model=model,
        train_dataset=tok_train,
        eval_dataset={"id": tok_id, "ood": tok_ood},
        args=TrainingArguments(
            output_dir=f"./ckpt/scan_sft_{config_name}",
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            num_train_epochs=EPOCHS, learning_rate=LR,
            logging_steps=20, logging_first_step=True,
            eval_strategy="steps", eval_steps=100,
            label_smoothing_factor=cfg["label_smoothing"],
            save_strategy="no", eval_on_start=True,
            report_to="none", bf16=True,
            dataloader_pin_memory=False, seed=SEED,
        ),
        callbacks=[acc_cb],
    )
    t.train()

    model.eval()
    acc_id = compute_scan_accuracy(model, tokenizer, test_id, 200)
    acc_ood = compute_scan_accuracy(model, tokenizer, test_ood, 200)
    results[config_name] = {"id_acc": acc_id, "ood_acc": acc_ood}
    all_dfs[config_name] = aggregate_log_history(t.state.log_history)

    del model, tokenizer, t
    torch.cuda.empty_cache()
    print(f"  → ID: {acc_id:.2%} | OOD: {acc_ood:.2%}")

print(f"\n{'='*60}")
print("Final Results:")
print(f"{'='*60}")
for name, r in results.items():
    print(f"  {name:15s}  ID={r['id_acc']:.2%}  OOD={r['ood_acc']:.2%}")

In [ ]:
# Compare Configs — Loss + Accuracy + Eval MBE
colors = {"sft": "gray", "sft_ls": "blue", "sft_dropout": "orange", "sft_mbe": "red", "sft_frob": "green", "sft_spectral": "purple"}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f"SFT Config Comparison — SCAN ({MODEL_NAME})", fontsize=14)

for key, title, ax in [("loss", "Train Loss", axes[0]), ("eval_id_acc", "ID Accuracy", axes[1]), ("eval_ood_acc", "OOD Accuracy", axes[2])]:
    for name, _df in all_dfs.items():
        if key in _df.columns and _df[key].notna().any():
            ax.plot(_df["step"], _df[key], label=name, color=colors.get(name, "black"), linewidth=0.8)
    ax.set_title(title); ax.set_xlabel("step"); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    if "acc" in key:
        ax.set_ylim(-0.05, 1.05)

plt.tight_layout(); plt.show()

# Eval MBE comparison across all configs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Eval MBE Comparison — Does MBE ↑ or ↓ during SFT? ({MODEL_NAME})", fontsize=14)

for ax, prefix, title in [
    (axes[0], "eval_id", "ID Eval MBE"),
    (axes[1], "eval_ood", "OOD Eval MBE"),
]:
    mbe_col = f"{prefix}_mbe_val"
    for name, _df in all_dfs.items():
        if mbe_col in _df.columns and _df[mbe_col].notna().any():
            vals = _df[mbe_col].dropna()
            steps = _df.loc[vals.index, "step"]
            ax.plot(steps, vals, "o-", color=colors.get(name, "black"),
                    linewidth=1.2, markersize=3, label=name)
    ax.set_title(title); ax.set_xlabel("step"); ax.set_ylabel("MBE")
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
names = list(results.keys())
x = np.arange(len(names))
ax.bar(x - 0.2, [results[n]["id_acc"] for n in names], 0.35, label="ID", color="steelblue")
ax.bar(x + 0.2, [results[n]["ood_acc"] for n in names], 0.35, label="OOD", color="coral")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15)
ax.set_ylabel("Accuracy"); ax.set_ylim(0, 1.05)
ax.legend(); ax.set_title(f"SCAN Accuracy — {MODEL_NAME}")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

In [ ]:
# (end of notebook)